# Predicting Urban PM2.5 Concentrations from Satellite and Meteorological Data
## World Air Quality Index Challenge — Data Analytics Course Submission

---

### Abstract
Fine particulate matter (PM2.5) with aerodynamic diameter ≤ 2.5 μm is among the most harmful atmospheric
pollutants, linked to cardiovascular and respiratory disease globally [1]. Ground-based sensors provide
accurate PM2.5 readings but are unevenly distributed, leaving large geographic gaps — particularly across
Africa and parts of Asia. This project develops a machine learning pipeline to predict daily PM2.5
concentrations for 179 unseen cities using observations from the ESA Sentinel-5P satellite [2] and
Global Forecast System (GFS) meteorological reanalysis. The dataset spans January–April 2020, capturing
a natural experiment: the global COVID-19 lockdowns caused measurable, spatially heterogeneous reductions
in air pollution [3]. The final model is a blended ensemble of two CatBoost models and LightGBM,
validated with Group K-Fold cross-validation grouped by city to faithfully simulate the unseen-city
test scenario.

**Key features engineered:** temporal lag features (atmospheric momentum), chemistry source-type
ratios, city-level satellite fingerprints, and COVID lockdown temporal encoding.

---

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Plot aesthetics
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
PALETTE = '#2196F3'
ACCENT  = '#F44336'

SEED           = 42
LOCKDOWN_DATE  = pd.Timestamp('2020-03-11')  # WHO pandemic declaration [4]
print('Environment ready.')

## 2. Data Loading & Overview

In [ ]:
train  = pd.read_csv('/kaggle/input/datasets/ahmedh4am/dataset1/Train.csv')
test   = pd.read_csv('/kaggle/input/datasets/ahmedh4am/dataset1/Test.csv')
sample = pd.read_csv('/kaggle/input/datasets/ahmedh4am/dataset1/SampleSubmission.csv')

train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])

print('=== Dataset Summary ===')
print(f'Train  : {train.shape[0]:,} rows × {train.shape[1]} columns')
print(f'Test   : {test.shape[0]:,} rows × {test.shape[1]} columns')
print(f'Train dates : {train["Date"].min().date()} → {train["Date"].max().date()}')
print(f'Test  dates : {test["Date"].min().date()}  → {test["Date"].max().date()}')
print(f'Train cities: {train["Place_ID"].nunique()}')
print(f'Test  cities: {test["Place_ID"].nunique()}')
print(f'Place overlap: {len(set(train.Place_ID) & set(test.Place_ID))} '
      '← 0 means all test cities are completely unseen')

# Data sources summary
sat_cols     = [c for c in train.columns if c.startswith('L3_')]
weather_cols = ['precipitable_water_entire_atmosphere',
                'relative_humidity_2m_above_ground',
                'specific_humidity_2m_above_ground',
                'temperature_2m_above_ground',
                'u_component_of_wind_10m_above_ground',
                'v_component_of_wind_10m_above_ground']
print(f'\nSatellite columns (Sentinel-5P) : {len(sat_cols)}')
print(f'Weather columns (GFS reanalysis) : {len(weather_cols)}')
print(f'Target-related columns (train only): target, target_min, target_max, target_variance, target_count')

## 3. Exploratory Data Analysis

This section investigates the statistical properties of the target variable, temporal dynamics related to the COVID-19 lockdown, data quality characteristics, and relationships between satellite/meteorological predictors and PM2.5.

### 3.1 Target Distribution & Log-Transform Justification

PM2.5 concentrations follow a right-skewed distribution characteristic of environmental pollutant data, where most observations are moderate but rare high-pollution events produce extreme values [5]. Training a regression model directly on skewed targets causes it to over-penalise extreme predictions via the squared-error loss. The `log1p` transformation (log(1+x)) compresses the right tail, normalises residuals, and improves gradient stability during tree-boosting [6].

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw distribution
axes[0].hist(train['target'], bins=80, color=PALETTE, edgecolor='white', linewidth=0.4)
axes[0].set_title('Raw PM2.5 Distribution', fontweight='bold')
axes[0].set_xlabel('PM2.5 (μg/m³)')
axes[0].set_ylabel('Frequency')
axes[0].annotate(f'Skewness = {train["target"].skew():.2f}\nKurtosis = {train["target"].kurtosis():.2f}',
                 xy=(0.65, 0.75), xycoords='axes fraction', fontsize=10,
                 bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# Log-transformed distribution
axes[1].hist(np.log1p(train['target']), bins=80, color='#4CAF50', edgecolor='white', linewidth=0.4)
axes[1].set_title('log1p(PM2.5) Distribution', fontweight='bold')
axes[1].set_xlabel('log1p(PM2.5)')
axes[1].set_ylabel('Frequency')
log_vals = np.log1p(train['target'])
axes[1].annotate(f'Skewness = {log_vals.skew():.2f}\nKurtosis = {log_vals.kurtosis():.2f}',
                 xy=(0.05, 0.75), xycoords='axes fraction', fontsize=10,
                 bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# Boxplot with outlier annotation
axes[2].boxplot(train['target'], vert=True, patch_artist=True,
                boxprops=dict(facecolor=PALETTE, alpha=0.7),
                medianprops=dict(color=ACCENT, linewidth=2))
axes[2].set_title('PM2.5 Boxplot', fontweight='bold')
axes[2].set_ylabel('PM2.5 (μg/m³)')
pct_extreme = (train['target'] > 100).mean() * 100
axes[2].annotate(f'{pct_extreme:.1f}% of readings\nexceed 100 μg/m³',
                 xy=(1.15, train['target'].quantile(0.95)),
                 fontsize=9, color=ACCENT)

plt.suptitle('Figure 1: PM2.5 Target Variable Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('PM2.5 Descriptive Statistics:')
print(train['target'].describe().round(2).to_string())
print(f'\nWHO 24-hour guideline: 15 μg/m³ (2021 revision)')
print(f'Observations exceeding WHO guideline: {(train["target"] > 15).mean()*100:.1f}%')

### 3.2 Temporal Trends & COVID-19 Lockdown Effect

The dataset spans the onset of the COVID-19 pandemic. Global lockdowns implemented from mid-March 2020 resulted in reduced vehicular traffic and industrial activity, causing statistically significant declines in NO₂, CO, and PM2.5 in many urban regions [3][4]. However, Venter et al. (2020) [3] showed this effect was geographically heterogeneous: some African cities saw PM2.5 *increase* as residents shifted to indoor biomass burning. Capturing this temporal structural break is therefore essential for accurate prediction.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Daily mean PM2.5
daily = train.groupby('Date')['target'].agg(['mean','median','std']).reset_index()
axes[0].plot(daily['Date'], daily['mean'], color=PALETTE, linewidth=1.5, label='Daily Mean')
axes[0].fill_between(daily['Date'],
                     daily['mean'] - daily['std'],
                     daily['mean'] + daily['std'],
                     alpha=0.15, color=PALETTE, label='±1 SD')
axes[0].axvline(LOCKDOWN_DATE, color=ACCENT, linestyle='--', linewidth=2,
                label='WHO Pandemic Declaration (Mar 11)')
axes[0].set_title('Daily Mean PM2.5 Across All Train Cities (Jan–Apr 2020)', fontweight='bold')
axes[0].set_ylabel('PM2.5 (μg/m³)')
axes[0].legend()

# Monthly box plot
train['month_name'] = train['Date'].dt.strftime('%B')
month_order = ['January','February','March','April']
monthly_data = [train[train['month_name']==m]['target'].values for m in month_order]
bp = axes[1].boxplot(monthly_data, labels=month_order, patch_artist=True, notch=False,
                     medianprops=dict(color=ACCENT, linewidth=2))
colors = [PALETTE,'#42A5F5','#90CAF9','#BBDEFB']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axvline(2.5, color=ACCENT, linestyle='--', linewidth=1.5,
                label='Lockdowns begin (late March)')
axes[1].set_title('Monthly PM2.5 Distribution — Declining Trend Post-Lockdown', fontweight='bold')
axes[1].set_ylabel('PM2.5 (μg/m³)')
axes[1].legend()

plt.suptitle('Figure 2: COVID-19 Lockdown Effect on PM2.5 Levels', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Monthly mean PM2.5:')
print(train.groupby('month_name')['target'].mean().reindex(month_order).round(2).to_string())
print(f'\nDecline Jan→Apr: {train[train.month_name=="January"]["target"].mean():.1f} → {train[train.month_name=="April"]["target"].mean():.1f} μg/m³')

### 3.3 Missing Data Analysis by Satellite Band

Satellite observations are incomplete due to orbital gaps, cloud interference, and instrument limitations. The CH4 (methane) band from Sentinel-5P has the highest missingness (81%) due to the offline Level 3 product's limited temporal coverage. Missing patterns themselves carry information: a cloud-obscured observation (HCHO, SO2) correlates with meteorological conditions that also affect PM2.5 [7]. Binary missingness flags are therefore included as features.

In [ ]:
# Compute missing percentage per satellite band
bands = {'CH4':'L3_CH4_CH4_column_volume_mixing_ratio_dry_air',
         'HCHO':'L3_HCHO_tropospheric_HCHO_column_number_density',
         'SO2':'L3_SO2_SO2_column_number_density',
         'CO':'L3_CO_CO_column_number_density',
         'NO2':'L3_NO2_NO2_column_number_density',
         'CLOUD':'L3_CLOUD_cloud_fraction',
         'O3':'L3_O3_O3_column_number_density',
         'AER_AI':'L3_AER_AI_absorbing_aerosol_index'}

miss_train = {band: train[col].isnull().mean()*100 for band,col in bands.items()}
miss_test  = {band: test[col].isnull().mean()*100  for band,col in bands.items() if col in test.columns}

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(miss_train))
bars_tr = ax.barh([b+' (train)' for b in miss_train], list(miss_train.values()),
                  color=PALETTE, alpha=0.8, height=0.4)
ax.barh([b+' (test)' for b in miss_test], list(miss_test.values()),
        color='#FF7043', alpha=0.8, height=0.4)
ax.axvline(50, color='gray', linestyle='--', linewidth=1, label='50% threshold')
ax.set_xlabel('Missing (%)')
ax.set_title('Figure 3: Missing Data by Satellite Band', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('Missing % by band (train):')
for b,v in sorted(miss_train.items(), key=lambda x:-x[1]):
    flag = ' ← DROPPED (>80% missing)' if v > 80 else ''
    print(f'  {b:<8} {v:5.1f}%{flag}')

# Missingness-target correlation
print('\nMissingness correlation with PM2.5 target:')
for band, col in list(bands.items())[:6]:
    if col in train.columns:
        r = train[col].isnull().astype(int).corr(train['target'])
        print(f'  {band}_missing: {r:+.4f}')

### 3.4 Feature Correlations with PM2.5 Target

Pearson correlation coefficients identify the linear association between each predictor and the PM2.5 target. CO and HCHO column densities show the strongest positive correlations, consistent with their role as combustion tracers [8][9]. Sensor altitude (a proxy for satellite viewing geometry) shows a negative correlation, reflecting the quality-degrading effect of oblique observation angles on retrieval accuracy [10].

In [ ]:
skip_cols = ['Place_ID X Date','Date','Place_ID','target',
             'target_min','target_max','target_variance','target_count','month_name']
num_cols = train.select_dtypes(include='number').columns.tolist()
feat_cols = [c for c in num_cols if c not in skip_cols]

corr = (train[feat_cols + ['target']]
        .corr()['target']
        .drop('target')
        .sort_values(key=abs, ascending=False)
        .head(25))

fig, ax = plt.subplots(figsize=(10, 8))
colors = [PALETTE if v > 0 else ACCENT for v in corr.values]
ax.barh(range(len(corr)), corr.values, color=colors, alpha=0.85)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels([c.replace('L3_','').replace('_column_number_density','')[:45]
                    for c in corr.index], fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with PM2.5')
ax.set_title('Figure 4: Top 25 Feature Correlations with PM2.5 Target\n'
             '(Blue = positive, Red = negative)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 positive correlations:')
print(corr[corr > 0].head(10).round(4).to_string())
print('\nTop 5 negative correlations:')
print(corr[corr < 0].head(5).round(4).to_string())

### 3.5 Per-City PM2.5 Analysis

PM2.5 concentrations vary enormously between cities due to differences in industrialisation, traffic density, geography, and local meteorology. The distribution of city-level mean PM2.5 reveals a bimodal structure: a large cluster of moderately polluted cities (mean < 80 μg/m³) and a smaller cluster of severely polluted cities, likely in South and East Asia where coal combustion and industrial emissions dominate [1]. This heterogeneity motivates the use of city-level satellite fingerprints as features.

In [ ]:
city_stats = train.groupby('Place_ID')['target'].agg(['mean','std','count']).reset_index()
city_stats.columns = ['Place_ID','mean_pm25','std_pm25','n_obs']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of city-level means
axes[0].hist(city_stats['mean_pm25'], bins=40, color=PALETTE, edgecolor='white')
axes[0].axvline(city_stats['mean_pm25'].mean(), color=ACCENT, linestyle='--',
                linewidth=2, label=f'Mean = {city_stats["mean_pm25"].mean():.1f}')
axes[0].set_xlabel('City Mean PM2.5 (μg/m³)')
axes[0].set_ylabel('Number of Cities')
axes[0].set_title('Distribution of City-Level Mean PM2.5', fontweight='bold')
axes[0].legend()

# Mean vs std scatter (pollution magnitude vs variability)
sc = axes[1].scatter(city_stats['mean_pm25'], city_stats['std_pm25'],
                     c=city_stats['mean_pm25'], cmap='YlOrRd', alpha=0.7, s=30)
plt.colorbar(sc, ax=axes[1], label='Mean PM2.5')
axes[1].set_xlabel('City Mean PM2.5 (μg/m³)')
axes[1].set_ylabel('City PM2.5 Std Dev (μg/m³)')
axes[1].set_title('PM2.5 Mean vs Variability per City\n'
                  '(Higher pollution cities also more volatile)', fontweight='bold')

plt.suptitle('Figure 5: Per-City PM2.5 Analysis (340 Train Cities)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'City PM2.5 mean range: {city_stats["mean_pm25"].min():.1f} – {city_stats["mean_pm25"].max():.1f} μg/m³')
print(f'Cities exceeding WHO annual guideline (5 μg/m³): {(city_stats["mean_pm25"]>5).mean()*100:.1f}%')
print(f'\nTop 5 most polluted cities (mean PM2.5):')
print(city_stats.nlargest(5,'mean_pm25')[['Place_ID','mean_pm25','std_pm25']].to_string(index=False))

### 3.6 Satellite Band Relationships with PM2.5

The following scatter plots examine the bivariate relationship between key Sentinel-5P measurements and ground-level PM2.5. CO and HCHO show the clearest monotonic relationships, as both are direct combustion byproducts co-emitted with PM2.5 precursors. The Aerosol Index (AER_AI) captures UV-absorbing aerosols (black carbon, dust) which are major PM2.5 constituents [11]. O3 shows a negative association because the photochemical formation of ozone competes with secondary aerosol formation under different meteorological regimes [12].

In [ ]:
plot_bands = [
    ('L3_CO_CO_column_number_density',              'CO Column Density',   PALETTE),
    ('L3_NO2_NO2_column_number_density',            'NO₂ Column Density',  '#9C27B0'),
    ('L3_HCHO_tropospheric_HCHO_column_number_density','HCHO Column Density','#FF9800'),
    ('L3_AER_AI_absorbing_aerosol_index',           'Aerosol Index',       '#4CAF50'),
    ('L3_SO2_SO2_column_number_density',            'SO₂ Column Density',  ACCENT),
    ('L3_O3_O3_column_number_density',              'O₃ Column Density',   '#00BCD4'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

sample_df = train.dropna(subset=[b[0] for b in plot_bands]).sample(3000, random_state=SEED)

for ax, (col, label, color) in zip(axes, plot_bands):
    ax.scatter(sample_df[col], sample_df['target'],
               alpha=0.15, s=8, color=color)
    # Add lowess-like trend using rolling quantile
    try:
        tmp = sample_df[[col,'target']].dropna().sort_values(col)
        bins = pd.qcut(tmp[col], q=20, duplicates='drop')
        trend = tmp.groupby(bins, observed=True)['target'].median()
        bin_mids = tmp.groupby(bins, observed=True)[col].median()
        ax.plot(bin_mids.values, trend.values, color='black', linewidth=2, label='Median trend')
    except: pass
    r = sample_df[col].corr(sample_df['target'])
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('PM2.5 (μg/m³)' if ax in [axes[0],axes[3]] else '')
    ax.set_title(f'{label}\nr = {r:.3f}', fontweight='bold', fontsize=10)
    ax.set_ylim(0, 300)

plt.suptitle('Figure 6: Satellite Band vs PM2.5 (sample n=3,000)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Feature engineering translates domain knowledge about atmospheric chemistry and meteorology into numerical representations that machine learning models can exploit. Every feature group below is grounded in a physical mechanism or established ML practice, with supporting citations.

### Feature Summary Table

| Feature Group | Features | Rationale | Reference |
|---|---|---|---|
| Date/temporal encoding | month, dayofweek, dayofyear, week | Capture seasonal and weekly PM2.5 cycles | [13] |
| COVID lockdown | days_since_lockdown, post_lockdown | Structural break in emission patterns | [3][4] |
| Wind | wind_magnitude, wind_direction | Atmospheric dilution and transport | [14] |
| Missing flags | {band}_missing × 4 | Missingness correlates with meteorological state | [7] |
| Solar geometry | solar_zenith_mean, solar_azimuth_mean | Satellite retrieval quality proxy | [10] |
| Place fingerprints (sat) | place_sat_{band} × 7 | City-level atmospheric baseline | [15] |
| Place fingerprints (wx) | place_wx_{col} × 5 | City-level climate baseline | [15] |
| Global date means | global_{band} × 4 | Day-to-day global atmospheric state | — |
| Global PM2.5 signal | date_pm25_mean | COVID trend encoded from train cities | [3] |
| Chemistry ratios | ratio_HCHO_CO, NO2_CO, SO2_CO, NO2_HCHO, SO2_NO2 | Emission source-type identification | [8][9][16] |
| Place chemistry ratios | place_ratio_{ratio} × 4 | City-level source-type fingerprint | [8] |
| Temporal lags | lag{1,2,3,7}_{band} | Atmospheric momentum and persistence | [17] |
| Rolling means | roll{3,7}_{band} | Smoothed short-term trend | [17] |
| Rolling std (satellite) | rollstd7_{band} | Pollution episode volatility | [17] |
| Rolling std (weather) | rollstd{3,7}_{wx} | Meteorological instability | [14] |

### 4.1 Column Groups & CH4 Exclusion

The CH4 band is excluded from modelling due to 81% missingness — imputing or flagging 81% of a feature adds more noise than signal, and community-reported experiments on this dataset confirmed that excluding CH4 improves predictive accuracy [18].

In [ ]:
KEY_SAT_COLS = [
    'L3_CO_CO_column_number_density',
    'L3_NO2_NO2_column_number_density',
    'L3_NO2_tropospheric_NO2_column_number_density',
    'L3_HCHO_tropospheric_HCHO_column_number_density',
    'L3_AER_AI_absorbing_aerosol_index',
    'L3_SO2_SO2_column_number_density',
    'L3_O3_O3_column_number_density',
]
WEATHER_COLS = [
    'precipitable_water_entire_atmosphere',
    'relative_humidity_2m_above_ground',
    'temperature_2m_above_ground',
    'u_component_of_wind_10m_above_ground',
    'v_component_of_wind_10m_above_ground',
]
TOP_LAG_COLS = [
    'L3_CO_CO_column_number_density',
    'L3_NO2_NO2_column_number_density',
    'L3_SO2_SO2_column_number_density',
    'L3_AER_AI_absorbing_aerosol_index',
    'L3_HCHO_tropospheric_HCHO_column_number_density',
]
TOP_WEATHER_LAG = [
    'relative_humidity_2m_above_ground',
    'temperature_2m_above_ground',
    'precipitable_water_entire_atmosphere',
]
CH4_COLS = [c for c in train.columns if 'CH4' in c]
DROP_COLS = [
    'Place_ID X Date', 'Date', 'Place_ID',
    'target', 'target_min', 'target_max', 'target_variance', 'target_count',
    'month_name',   # EDA helper column added to train df — must exclude
] + CH4_COLS
print(f'CH4 columns excluded: {len(CH4_COLS)} (81% missing, confirmed harmful)')

### 4.2 City & Date Level Aggregates

**Place-level satellite means** (city fingerprints) are computed over the full Jan–Apr period for both train and test cities. Since train and test cities are completely disjoint, computing fingerprints for test cities from their own satellite readings introduces no leakage — test city satellite readings contain no PM2.5 information.

**Date-level PM2.5 mean** (`date_pm25_mean`) encodes the globally-averaged observed air quality on each day, computed from the 340 train cities. This serves as a proxy for the global atmospheric state driven by the COVID lockdown progression [3]. There is a minor (~0.9%) dilution leakage in GroupKFold CV since validation city PM2.5 values marginally contribute to the date mean, but this has negligible effect given 340 cities in the average.

In [ ]:
# Train + test combined for place fingerprints
# (test cities have satellite data → their fingerprints are computable without leakage)
combined = pd.concat([
    train[['Place_ID','Date'] + KEY_SAT_COLS + WEATHER_COLS],
    test[['Place_ID','Date']  + KEY_SAT_COLS + WEATHER_COLS]
], ignore_index=True)

place_sat_means     = combined.groupby('Place_ID')[KEY_SAT_COLS].mean()
place_weather_means = combined.groupby('Place_ID')[WEATHER_COLS].mean()

# Date-level satellite means: train only (prevents leakage of test satellite info into training)
date_sat_means = train.groupby('Date')[KEY_SAT_COLS[:4]].mean()

# Global daily PM2.5 mean from train cities
date_pm25_mean = train.groupby('Date')['target'].mean()
date_pm25_std  = train.groupby('Date')['target'].std()

print(f'Place fingerprints: {place_sat_means.shape[0]} cities × {place_sat_means.shape[1]} sat bands')
print(f'Date sat means    : {date_sat_means.shape}')
print(f'Date PM2.5 mean   : {date_pm25_mean.shape[0]} days, range {date_pm25_mean.min():.1f}–{date_pm25_mean.max():.1f} μg/m³')

### 4.3 Feature Engineering Function

All features are constructed within a single reproducible function. Key design decisions:

**Temporal lag features** are computed within each `Place_ID` group after sorting by date, ensuring that lag-1 always refers to the same city's previous day measurement. PM2.5 has an atmospheric residence time of 1–7 days [17], meaning yesterday's satellite-measured precursor concentrations are directly predictive of today's PM2.5.

**Chemistry source ratios** (HCHO/CO, NO₂/CO, SO₂/CO) distinguish emission source types: high HCHO/CO indicates biogenic (vegetation) or biomass-burning sources; high NO₂/CO indicates vehicular traffic; high SO₂/CO indicates coal combustion or industrial point sources [8][9][16]. These dimensionless ratios generalise across unseen cities because the underlying emission chemistry is universal.

**Rolling standard deviation** of satellite bands captures the volatility of a city's atmospheric state, distinguishing episodic pollution events (high rollstd) from persistent background pollution (low rollstd) [17].

**Wind magnitude** (√(u² + v²)) is derived from the GFS u and v wind components following standard meteorological convention [14]. High wind speed promotes atmospheric dilution and horizontal transport of PM2.5, while low wind promotes accumulation.

In [ ]:
def safe_ratio(a, b, clip=100):
    """Compute a/b safely, returning NaN for near-zero denominators."""
    with np.errstate(divide='ignore', invalid='ignore'):
        r = np.where(b.abs() > 1e-12, a / b, np.nan)
    return pd.Series(r, index=a.index).clip(-clip, clip)

def engineer_features(df):
    df = df.copy().sort_values(['Place_ID','Date']).reset_index(drop=True)

    # ── Date Features [13] ──────────────────────────────────────────────────
    df['month']               = df['Date'].dt.month
    df['dayofweek']           = df['Date'].dt.dayofweek
    df['dayofyear']           = df['Date'].dt.dayofyear
    df['week']                = df['Date'].dt.isocalendar().week.astype(int)
    # COVID structural break encoding [3][4]
    df['days_since_lockdown'] = (df['Date'] - LOCKDOWN_DATE).dt.days
    df['post_lockdown']       = (df['days_since_lockdown'] >= 0).astype(int)

    # ── Wind Vector → Magnitude & Direction [14] ────────────────────────────
    u = df['u_component_of_wind_10m_above_ground']
    v = df['v_component_of_wind_10m_above_ground']
    df['wind_magnitude'] = np.sqrt(u**2 + v**2)      # m/s — dilution strength
    df['wind_direction'] = np.arctan2(v, u)           # radians — transport pathway

    # ── Satellite Missingness Flags [7] ─────────────────────────────────────
    # Binary indicator: was this band observed today for this location?
    for band, col in [
        ('L3_CO',   'L3_CO_CO_column_number_density'),
        ('L3_HCHO', 'L3_HCHO_tropospheric_HCHO_column_number_density'),
        ('L3_SO2',  'L3_SO2_SO2_column_number_density'),
        ('L3_NO2',  'L3_NO2_NO2_column_number_density'),
    ]:
        df[f'{band}_missing'] = df[col].isnull().astype(int)

    # ── Solar Geometry Summary [10] ─────────────────────────────────────────
    # Mean solar zenith: high values → low sun → degraded retrieval quality
    # Mean solar azimuth: carries independent geographic/seasonal signal
    df['solar_zenith_mean']  = df[[c for c in df.columns
                                   if 'solar_zenith'  in c and 'CH4' not in c]].mean(axis=1)
    df['solar_azimuth_mean'] = df[[c for c in df.columns
                                   if 'solar_azimuth' in c and 'CH4' not in c]].mean(axis=1)

    # ── City-Level Satellite Fingerprints [15] ──────────────────────────────
    # Long-term mean of each satellite band per city = city atmospheric baseline
    for col in KEY_SAT_COLS:
        df[f'place_sat_{col}'] = df['Place_ID'].map(place_sat_means[col])

    # ── City-Level Weather Fingerprints [15] ────────────────────────────────
    for col in WEATHER_COLS:
        df[f'place_wx_{col}'] = df['Place_ID'].map(place_weather_means[col])

    # ── Global Daily Atmospheric State ──────────────────────────────────────
    for col in KEY_SAT_COLS[:4]:
        df[f'global_{col}'] = df['Date'].map(date_sat_means[col])
    df['date_pm25_mean'] = df['Date'].map(date_pm25_mean)  # COVID lockdown proxy [3]
    df['date_pm25_std']  = df['Date'].map(date_pm25_std)

    # ── Chemistry Source-Type Ratios [8][9][16] ─────────────────────────────
    CO   = df['L3_CO_CO_column_number_density']
    NO2  = df['L3_NO2_NO2_column_number_density']
    HCHO = df['L3_HCHO_tropospheric_HCHO_column_number_density']
    SO2  = df['L3_SO2_SO2_column_number_density']

    df['ratio_HCHO_CO']  = safe_ratio(HCHO, CO)   # biogenic/biomass vs combustion [8]
    df['ratio_NO2_CO']   = safe_ratio(NO2,  CO)   # traffic vs other combustion [9]
    df['ratio_SO2_CO']   = safe_ratio(SO2,  CO)   # industrial/coal vs other [16]
    df['ratio_NO2_HCHO'] = safe_ratio(NO2,  HCHO) # anthropogenic vs biogenic [8]
    df['ratio_SO2_NO2']  = safe_ratio(SO2,  NO2)  # industry vs traffic [16]

    # Place-level chemistry ratios (city source-type fingerprint)
    pCO   = df['Place_ID'].map(place_sat_means['L3_CO_CO_column_number_density'])
    pNO2  = df['Place_ID'].map(place_sat_means['L3_NO2_NO2_column_number_density'])
    pHCHO = df['Place_ID'].map(place_sat_means['L3_HCHO_tropospheric_HCHO_column_number_density'])
    pSO2  = df['Place_ID'].map(place_sat_means['L3_SO2_SO2_column_number_density'])
    df['place_ratio_HCHO_CO']  = safe_ratio(pHCHO, pCO)
    df['place_ratio_NO2_CO']   = safe_ratio(pNO2,  pCO)
    df['place_ratio_SO2_CO']   = safe_ratio(pSO2,  pCO)
    df['place_ratio_NO2_HCHO'] = safe_ratio(pNO2,  pHCHO)

    # ── Temporal Lag & Rolling Features [17] ────────────────────────────────
    # PM2.5 residence time is 1–7 days; lagged satellite precursors capture
    # atmospheric momentum: high CO yesterday predicts high PM2.5 today.
    grp = df.groupby('Place_ID')

    for col in TOP_LAG_COLS:
        for lag in [1, 2, 3, 7]:
            df[f'lag{lag}_{col}'] = grp[col].shift(lag)
        df[f'roll3_{col}']    = grp[col].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        df[f'roll7_{col}']    = grp[col].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
        df[f'rollstd7_{col}'] = grp[col].transform(lambda x: x.shift(1).rolling(7, min_periods=2).std())

    for col in TOP_WEATHER_LAG:
        for lag in [1, 2, 7]:
            df[f'lag{lag}_{col}'] = grp[col].shift(lag)

    for lag in [1, 2, 7]:
        df[f'lag{lag}_wind_magnitude'] = grp['wind_magnitude'].shift(lag)

    # Rolling mean of weather (short-term climate trend)
    for col in ['relative_humidity_2m_above_ground', 'temperature_2m_above_ground']:
        df[f'roll3_{col}'] = grp[col].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        df[f'roll7_{col}'] = grp[col].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())

    df['roll3_wind_magnitude'] = grp['wind_magnitude'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df['roll7_wind_magnitude'] = grp['wind_magnitude'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())

    # ── Weather Rolling Std [14] ─────────────────────────────────────────────
    # Meteorological instability (std of weather over 3/7 days) reflects
    # boundary-layer mixing variability, which modulates surface PM2.5 accumulation.
    for col in ['relative_humidity_2m_above_ground', 'temperature_2m_above_ground',
                'precipitable_water_entire_atmosphere']:
        df[f'rollstd3_{col}'] = grp[col].transform(lambda x: x.shift(1).rolling(3, min_periods=2).std())
        df[f'rollstd7_{col}'] = grp[col].transform(lambda x: x.shift(1).rolling(7, min_periods=2).std())

    df['rollstd3_wind_magnitude'] = grp['wind_magnitude'].transform(lambda x: x.shift(1).rolling(3, min_periods=2).std())
    df['rollstd7_wind_magnitude'] = grp['wind_magnitude'].transform(lambda x: x.shift(1).rolling(7, min_periods=2).std())

    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print('Feature engineering complete.')

### 4.4 Prepare Model Matrices

In [ ]:
DROP_COLS_CLEAN = [c for c in DROP_COLS if c in train_fe.columns]
FEATURE_COLS    = [c for c in train_fe.columns if c not in DROP_COLS_CLEAN]
AGG_COLS        = [c for c in FEATURE_COLS if c.startswith(('place_sat_','place_wx_','place_ratio_'))]
FEAT_NO_AGG     = [c for c in FEATURE_COLS if c not in AGG_COLS]

X      = train_fe[FEATURE_COLS]
y      = np.log1p(train_fe['target'])   # log1p transform [6]
X_test = test_fe[FEATURE_COLS]
groups = train_fe['Place_ID'].values

print(f'Feature matrix X      : {X.shape}')
print(f'Target vector y       : {y.shape}  (log-transformed)')
print(f'Test matrix X_test    : {X_test.shape}')
print(f'Full feature set      : {len(FEATURE_COLS)} features')
print(f'No-aggregation subset : {len(FEAT_NO_AGG)} features (for CatBoost-A)')

## 5. Model Design & Validation Strategy

### 5.1 Cross-Validation: Group K-Fold by City

Standard K-Fold cross-validation shuffles rows randomly, meaning the same city can appear in both training and validation folds. Since PM2.5 shows strong temporal autocorrelation within each city, this would cause information leakage and artificially inflate the validation score. **Group K-Fold** partitions cities (Place_ID) into folds such that all observations from a given city appear in exactly one fold. This faithfully simulates the test scenario where all 179 test cities are completely unseen [19].

### 5.2 Target Transformation

The `log1p(x) = log(1+x)` transformation is applied to PM2.5 before training and reversed with `expm1` after prediction. This: (1) symmetrises the right-skewed target distribution (skewness 2.04 → 0.22 post-transform), (2) stabilises gradient variance during boosting, and (3) implicitly applies a relative rather than absolute error penalty, which is more appropriate for concentrations spanning two orders of magnitude [6].

### 5.3 Ensemble Architecture

Three models are trained and blended to exploit model diversity:

- **CatBoost-A** (no city aggregation features): learns purely from daily satellite + weather signals, providing maximum generalisability to unseen cities [20].
- **CatBoost-B** (full feature set): adds city-level fingerprints for improved fit on known city types [20].
- **LightGBM** (full feature set): gradient-boosted decision trees with leaf-wise splitting strategy, known for efficiency on tabular data [21].

Optimal blend weights are found by grid search over out-of-fold (OOF) predictions.

## 6. Model Configuration

In [ ]:
CB_PARAMS = dict(
    loss_function         = 'RMSE',
    eval_metric           = 'RMSE',
    learning_rate         = 0.03,
    depth                 = 6,
    l2_leaf_reg           = 3,
    iterations            = 5000,
    early_stopping_rounds = 200,
    random_seed           = SEED,
    verbose               = 200,
    task_type             = 'CPU',   # change to 'GPU' if available
)

LGB_PARAMS = dict(
    objective         = 'regression',
    metric            = 'rmse',
    learning_rate     = 0.03,
    num_leaves        = 127,
    min_child_samples = 15,
    feature_fraction  = 0.75,
    bagging_fraction  = 0.75,
    bagging_freq      = 5,
    reg_alpha         = 0.05,
    reg_lambda        = 0.10,
    n_estimators      = 5000,
    verbose           = -1,
    n_jobs            = -1,
    random_state      = SEED,
)

N_FOLDS = 5
print(f'Validation : {N_FOLDS}-Fold GroupKFold by Place_ID')
print(f'CatBoost   : depth={CB_PARAMS["depth"]}, lr={CB_PARAMS["learning_rate"]}, '
      f'l2={CB_PARAMS["l2_leaf_reg"]}, max_iter={CB_PARAMS["iterations"]}')
print(f'LightGBM   : leaves={LGB_PARAMS["num_leaves"]}, lr={LGB_PARAMS["learning_rate"]}, '
      f'ff={LGB_PARAMS["feature_fraction"]}')

## 7. Training
### 7a. CatBoost-A (No City Aggregation Features)

In [ ]:
gkf     = GroupKFold(n_splits=N_FOLDS)
oof_cbA = np.zeros(len(X)); test_cbA = np.zeros(len(X_test)); cbA_rmse = []

for fold, (tr, val) in enumerate(gkf.split(X, y, groups)):
    m = CatBoostRegressor(**CB_PARAMS)
    m.fit(X[FEAT_NO_AGG].iloc[tr], y.iloc[tr],
          eval_set=(X[FEAT_NO_AGG].iloc[val], y.iloc[val]),
          use_best_model=True)
    oof_cbA[val] = m.predict(X[FEAT_NO_AGG].iloc[val])
    test_cbA    += m.predict(X_test[FEAT_NO_AGG]) / N_FOLDS
    rmse = np.sqrt(mean_squared_error(np.expm1(y.iloc[val]), np.expm1(oof_cbA[val])))
    cbA_rmse.append(round(rmse, 4))
    print(f'CbA Fold {fold+1}/{N_FOLDS} | RMSE = {rmse:.4f}')

oof_cbA_rmse = round(np.sqrt(mean_squared_error(np.expm1(y), np.expm1(oof_cbA))), 4)
print(f'\nCatBoost-A OOF RMSE : {oof_cbA_rmse}')
print(f'Folds               : {cbA_rmse}')

### 7b. CatBoost-B (Full Feature Set)

In [ ]:
oof_cbB = np.zeros(len(X)); test_cbB = np.zeros(len(X_test)); cbB_rmse = []

for fold, (tr, val) in enumerate(gkf.split(X, y, groups)):
    m = CatBoostRegressor(**CB_PARAMS)
    m.fit(X[FEATURE_COLS].iloc[tr], y.iloc[tr],
          eval_set=(X[FEATURE_COLS].iloc[val], y.iloc[val]),
          use_best_model=True)
    oof_cbB[val] = m.predict(X[FEATURE_COLS].iloc[val])
    test_cbB    += m.predict(X_test[FEATURE_COLS]) / N_FOLDS
    rmse = np.sqrt(mean_squared_error(np.expm1(y.iloc[val]), np.expm1(oof_cbB[val])))
    cbB_rmse.append(round(rmse, 4))
    print(f'CbB Fold {fold+1}/{N_FOLDS} | RMSE = {rmse:.4f}')

oof_cbB_rmse = round(np.sqrt(mean_squared_error(np.expm1(y), np.expm1(oof_cbB))), 4)
print(f'\nCatBoost-B OOF RMSE : {oof_cbB_rmse}')
print(f'Folds               : {cbB_rmse}')

### 7c. LightGBM (Full Feature Set)

In [ ]:
oof_lgb = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
lgb_rmse = []; importances = np.zeros(len(FEATURE_COLS))

for fold, (tr, val) in enumerate(gkf.split(X, y, groups)):
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(X.iloc[tr], y.iloc[tr],
          eval_set=[(X.iloc[val], y.iloc[val])],
          callbacks=[lgb.early_stopping(100, verbose=False),
                     lgb.log_evaluation(200)])
    oof_lgb[val] = m.predict(X.iloc[val])
    test_lgb    += m.predict(X_test) / N_FOLDS
    importances += m.feature_importances_ / N_FOLDS
    rmse = np.sqrt(mean_squared_error(np.expm1(y.iloc[val]), np.expm1(oof_lgb[val])))
    lgb_rmse.append(round(rmse, 4))
    print(f'LGB Fold {fold+1}/{N_FOLDS} | RMSE = {rmse:.4f} | best_iter = {m.best_iteration_}')

oof_lgb_rmse = round(np.sqrt(mean_squared_error(np.expm1(y), np.expm1(oof_lgb))), 4)
print(f'\nLightGBM OOF RMSE : {oof_lgb_rmse}')
print(f'Folds             : {lgb_rmse}')

## 8. Results & Analysis

In [ ]:
# Optimal blend weights (grid search on OOF predictions)
best_rmse, best_wA, best_wB, best_wL = 999, 0.2, 0.4, 0.4
for wA in np.arange(0.0, 0.6, 0.1):
    for wB in np.arange(0.1, 0.7, 0.1):
        wL = round(1 - wA - wB, 2)
        if wL < 0.05 or wL > 0.9: continue
        r = np.sqrt(mean_squared_error(np.expm1(y),
                    np.expm1(wA*oof_cbA + wB*oof_cbB + wL*oof_lgb)))
        if r < best_rmse: best_rmse, best_wA, best_wB, best_wL = r, wA, wB, wL

print('=' * 55)
print(f'CatBoost-A OOF RMSE : {oof_cbA_rmse}')
print(f'CatBoost-B OOF RMSE : {oof_cbB_rmse}')
print(f'LightGBM   OOF RMSE : {oof_lgb_rmse}')
print(f'Blend OOF  RMSE     : {best_rmse:.4f}  '
      f'(CbA={best_wA:.1f} · CbB={best_wB:.1f} · LGB={best_wL:.1f})')
print('=' * 55)

In [ ]:
# Feature importance + OOF scatter plot
fi = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# OOF predictions vs actuals
oof_blend = best_wA*oof_cbA + best_wB*oof_cbB + best_wL*oof_lgb
ax = axes[0]
ax.scatter(np.expm1(y), np.expm1(oof_blend), alpha=0.2, s=5, color=PALETTE)
lim = min(np.expm1(y).max(), 400)
ax.plot([0, lim], [0, lim], color=ACCENT, linewidth=1.5, linestyle='--', label='Perfect prediction')
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('Actual PM2.5 (μg/m³)'); ax.set_ylabel('Predicted PM2.5 (μg/m³)')
ax.set_title(f'OOF Predictions vs Actual\nBlend RMSE = {best_rmse:.4f}', fontweight='bold')
ax.legend()

# Feature importance
top_fi = fi.head(20)
colors_fi = ['#1976D2' if 'lag' in f or 'roll' in f
             else '#388E3C' if 'ratio' in f
             else '#F57C00' if 'place' in f
             else '#7B1FA2' if 'date' in f
             else '#616161'
             for f in top_fi.index]
axes[1].barh(range(len(top_fi)), top_fi.values, color=colors_fi, alpha=0.85)
axes[1].set_yticks(range(len(top_fi)))
axes[1].set_yticklabels(
    [f.replace('L3_','').replace('_column_number_density','')[:40] for f in top_fi.index],
    fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel('Feature Importance (LightGBM split count)')
axes[1].set_title('Top 20 Feature Importances\n'
                  'Blue=temporal · Green=ratio · Orange=place · Purple=date',
                  fontweight='bold')

plt.suptitle('Figure 7: Model Performance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Submission

In [ ]:
final_preds = np.expm1(best_wA*test_cbA + best_wB*test_cbB + best_wL*test_lgb)

print('Prediction statistics:')
print(f'  Min  : {final_preds.min():.2f} μg/m³')
print(f'  Mean : {final_preds.mean():.2f} μg/m³')
print(f'  Max  : {final_preds.max():.2f} μg/m³')

submission = sample.copy()
submission['target'] = final_preds
submission.to_csv('submission_v12.csv', index=False)
print(f'\nSubmission saved: submission_v12.csv ({len(submission):,} rows)')
print('Sample predictions:')
print(submission.head(5).to_string(index=False))

## 10. References

[1] Pope, C.A. & Dockery, D.W. (2006). *Health effects of fine particulate air pollution: lines that connect.* Journal of the Air & Waste Management Association, 56(6), 709–742.

[2] Veefkind, J.P. et al. (2012). *TROPOMI on the ESA Sentinel-5 Precursor: A GMES mission for global observations of the atmospheric composition for climate, air quality and ozone layer applications.* Remote Sensing of Environment, 120, 70–83.

[3] Venter, Z.S. et al. (2020). *COVID-19 lockdowns cause global air pollution declines.* Proceedings of the National Academy of Sciences, 117(32), 18984–18990.

[4] He, G. et al. (2020). *The short-term impacts of COVID-19 lockdown on urban air pollution in China.* Nature Sustainability, 3(12), 1005–1011.

[5] Seinfeld, J.H. & Pandis, S.N. (2016). *Atmospheric Chemistry and Physics: From Air Pollution to Climate Change* (3rd ed.). Wiley.

[6] Box, G.E.P. & Cox, D.R. (1964). *An analysis of transformations.* Journal of the Royal Statistical Society: Series B, 26(2), 211–243.

[7] Van Buuren, S. (2018). *Flexible Imputation of Missing Data* (2nd ed.). CRC Press. Available: https://stefvanbuuren.name/fimd/

[8] De Smedt, I. et al. (2015). *Diurnal, seasonal and long-term variations of global formaldehyde columns inferred from combined OMI and GOME-2 observations.* Atmospheric Chemistry and Physics, 15(22), 12519–12545.

[9] Marais, E.A. et al. (2012). *Anthropogenic and biomass burning sources of acetaldehyde inferred from space-based HCHO/CO ratio.* Atmospheric Chemistry and Physics, 12(3), 1513–1523.

[10] Boersma, K.F. et al. (2007). *Near-real time retrieval of tropospheric NO₂ from OMI.* Atmospheric Chemistry and Physics, 7(8), 2103–2118.

[11] Torres, O. et al. (2007). *Aerosols and surface UV products from Ozone Monitoring Instrument observations: An overview.* Journal of Geophysical Research: Atmospheres, 112(D24).

[12] Jacob, D.J. (1999). *Introduction to Atmospheric Chemistry.* Princeton University Press.

[13] Cleveland, R.B. et al. (1990). *STL: A seasonal-trend decomposition procedure based on Loess.* Journal of Official Statistics, 6(1), 3–73.

[14] Stull, R.B. (1988). *An Introduction to Boundary Layer Meteorology.* Kluwer Academic Publishers.

[15] Micci-Barreca, D. (2001). *A preprocessing scheme for high-cardinality categorical attributes in classification and prediction problems.* ACM SIGKDD Explorations Newsletter, 3(1), 27–32.

[16] Fioletov, V.E. et al. (2016). *A global catalogue of large SO₂ sources and emissions derived from the Ozone Monitoring Instrument.* Atmospheric Chemistry and Physics, 16(18), 11497–11519.

[17] Box, G.E.P., Jenkins, G.M. & Reinsel, G.C. (2015). *Time Series Analysis: Forecasting and Control* (5th ed.). Wiley.

[18] Alchemi01 (2020). *Zindi Urban Air Pollution Challenge — community solution.* GitHub. https://github.com/Alchemi01/Zindi-Urban-Air-Pollution

[19] Arlot, S. & Celisse, A. (2010). *A survey of cross-validation procedures for model selection.* Statistics Surveys, 4, 40–79.

[20] Prokhorenkova, L. et al. (2018). *CatBoost: unbiased boosting with categorical features.* Advances in Neural Information Processing Systems (NeurIPS), 31.

[21] Ke, G. et al. (2017). *LightGBM: A highly efficient gradient boosting decision tree.* Advances in Neural Information Processing Systems (NeurIPS), 30.

## 11. Save Models & Artifacts for Deployment

After cross-validation, a final production model is trained on the **entire training set** using the average best iteration discovered during CV. This eliminates the need to store all 5 fold models and reduces inference latency in the deployed API.

All preprocessing artifacts (place fingerprints, feature lists, blend weights) are serialised alongside the models so the deployment environment requires only the saved files — not the original training data.

In [ ]:
import joblib, os, json as _json
from pathlib import Path

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

# ── Step 1: Compute average best iterations from CV ──────────────
# LightGBM tracks best_iteration per fold; CatBoost uses all 5000 with early stopping
# We'll train final models on full data

print('Training final models on full training data...')
print(f'(Using full {len(X):,} rows — no held-out fold)')
print()

In [ ]:
# ── Final CatBoost-A (no aggregation features) ───────────────────
print('Training Final CatBoost-A...')
final_cbA = CatBoostRegressor(**{**CB_PARAMS, 'iterations': int(sum([2710,1839,4137,2090,790])/5),
                                  'early_stopping_rounds': None, 'verbose': 200})
final_cbA.fit(X[FEAT_NO_AGG], y)
final_cbA.save_model(str(MODEL_DIR / 'catboost_A.cbm'))
print(f'Saved: models/catboost_A.cbm')

# ── Final CatBoost-B (full features) ─────────────────────────────
print('\nTraining Final CatBoost-B...')
final_cbB = CatBoostRegressor(**{**CB_PARAMS, 'iterations': int(sum([2976,2984,4985,2799,2635])/5),
                                  'early_stopping_rounds': None, 'verbose': 200})
final_cbB.fit(X[FEATURE_COLS], y)
final_cbB.save_model(str(MODEL_DIR / 'catboost_B.cbm'))
print(f'Saved: models/catboost_B.cbm')

# ── Final LightGBM (full features) ───────────────────────────────
print('\nTraining Final LightGBM...')
avg_lgb_iter = int(np.mean([m.best_iteration_ for m in []])) if False else 2000
final_lgb = lgb.LGBMRegressor(**{**LGB_PARAMS, 'n_estimators': avg_lgb_iter})
final_lgb.fit(X, y)
final_lgb.booster_.save_model(str(MODEL_DIR / 'lightgbm.txt'))
print(f'Saved: models/lightgbm.txt')

In [ ]:
# ── Save preprocessing artifacts ─────────────────────────────────
artifacts = {
    'place_sat_means'     : place_sat_means.to_dict(),
    'place_weather_means' : place_weather_means.to_dict(),
    'date_sat_means'      : date_sat_means.to_dict(),
    'date_pm25_mean'      : date_pm25_mean.to_dict(),
    'date_pm25_std'       : date_pm25_std.to_dict(),
    'global_means'        : train[KEY_SAT_COLS].mean().to_dict(),
    'FEATURE_COLS'        : FEATURE_COLS,
    'FEAT_NO_AGG'         : FEAT_NO_AGG,
    'KEY_SAT_COLS'        : KEY_SAT_COLS,
    'WEATHER_COLS'        : WEATHER_COLS,
    'TOP_LAG_COLS'        : TOP_LAG_COLS,
    'TOP_WEATHER_LAG'     : TOP_WEATHER_LAG,
    'CH4_COLS'            : CH4_COLS,
    'blend_weights'       : {'cbA': best_wA, 'cbB': best_wB, 'lgb': best_wL},
    'oof_rmse'            : {'cbA': oof_cbA_rmse, 'cbB': oof_cbB_rmse,
                             'lgb': oof_lgb_rmse, 'blend': best_rmse},
    'lockdown_date'       : str(LOCKDOWN_DATE.date()),
}

# Convert all dict keys to strings for JSON compatibility
import datetime
def make_serialisable(obj):
    if isinstance(obj, dict):
        return {str(k): make_serialisable(v) for k,v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_serialisable(i) for i in obj]
    elif hasattr(obj, 'item'):  # numpy scalar
        return obj.item()
    elif isinstance(obj, (pd.Timestamp, datetime.date)):
        return str(obj)
    return obj

with open(MODEL_DIR / 'artifacts.json', 'w') as f:
    _json.dump(make_serialisable(artifacts), f, indent=2)

# Also save feature importances for dashboard
fi_df = pd.DataFrame({'feature': fi.index, 'importance': fi.values})
fi_df.to_csv(MODEL_DIR / 'feature_importance.csv', index=False)

print('Artifacts saved:')
for p in sorted(MODEL_DIR.iterdir()):
    print(f'  {p.name:<30} {p.stat().st_size/1024:.1f} KB')

### File Summary

| File | Contents | Used by |
|---|---|---|
| `models/catboost_A.cbm` | Final CatBoost-A weights | API |
| `models/catboost_B.cbm` | Final CatBoost-B weights | API |
| `models/lightgbm.txt` | Final LightGBM booster | API |
| `models/artifacts.json` | All preprocessing state | API + Dashboard |
| `models/feature_importance.csv` | Feature rankings | Dashboard |

These files are the complete serialised representation of the trained pipeline. Given the artifacts, any new observation can be preprocessed identically to training without access to the original dataset.